# Time-series interpolation

**Purpose.** Merge predictors and interpolate gaps in weekly lake series.

**Inputs.** Weekly water-quality, ERA5-Land, FireCCI, and metadata tables.

**Outputs.** Per-lake interpolated weekly time series.

> Historical research notebook. Paths assume the repository layout described in `data/README.md`; generated outputs are intentionally not stored in the notebook.


In [ ]:
from pathlib import Path
import os

start_dir = Path.cwd().resolve()
for candidate in (start_dir, *start_dir.parents):
    if (candidate / 'notebooks').is_dir() and (candidate / 'README.md').is_file():
        os.chdir(candidate)
        break
else:
    raise RuntimeError('Run this notebook from inside the cloned repository.')


In [ ]:
import os
import glob
import pandas as pd

def interpolate_lake_csvs_with_control(input_folder, output_folder, method="linear", limit=4):
    os.makedirs(output_folder, exist_ok=True)

    for file in glob.glob(os.path.join(input_folder, "Lake_*.csv")):
        try:
            df = pd.read_csv(file)

            # Step 1: Limited linear interpolation
            numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns
            df[numeric_cols] = df[numeric_cols].interpolate(method=method, limit=limit, limit_direction='both')

            # Step 2: Fallback fill only for predictors
            fallback_fill = [
                "lswt_mean", "lake_mix_layer_temperature", "runoff_sum",
                "surface_runoff_sum", "temperature_2m", "total_precipitation_sum"
            ]
            for col in fallback_fill:
                if col in df.columns:
                    df[col] = df[col].ffill().bfill()

                    # Optional: fill remaining NaN with mean if nothing at all exists
                    if df[col].isna().all():
                        df[col] = df[col].fillna(0)  # or df[col].mean(), or other neutral default

            # Step 3: Save updated file
            lake_id = os.path.basename(file).split("_")[1].split(".")[0]
            out_path = os.path.join(output_folder, f"Lake_{lake_id}.csv")
            df.to_csv(out_path, index=False)
            print(f"Interpolated and saved: {out_path}")

        except Exception as e:
            print(f"Error processing {file}: {e}")


In [ ]:
interpolate_lake_csvs_with_control("Datasets/Merged_With_Metadata", "Datasets/Interpolated_Lake_CSVs")

In [ ]:
import os
import glob
import pandas as pd

def check_no_missing_values(folder):
    missing_report = []

    for file in glob.glob(os.path.join(folder, "Lake_*.csv")):
        df = pd.read_csv(file)
        if df.isna().any().any():  # if any NaN in entire DataFrame
            lake_id = os.path.basename(file).split("_")[1].split(".")[0]
            missing_report.append(lake_id)

    if missing_report:
        print(f"⚠️ Files with missing values after interpolation: {missing_report}")
        print(f"Total with missing values: {len(missing_report)}")
    else:
        print("✅ All files are complete. No missing values found.")

# ▶️ To run:
check_no_missing_values("Datasets/Interpolated_Lake_CSVs")
